In [1]:
from crawler import crawl_wikipedia
from dictionary import DictionaryBuilder
from indexing import DocumentIndexer, QueryIndexer
from similarity import rank_documents

from io_utils import (
    save_urls, load_urls, urls_exist,
    save_dictionary, load_dictionary, dictionary_exist,
)

## Crawling

In [2]:
test_url = "https://fr.wikipedia.org/wiki/L%27%C3%89vangile_du_monstre_en_spaghettis_volant"
depth = 2

print("=" * 50)

# ---- URL collection ----
if urls_exist():
    print("Loading URLs from cache...")
    urls = load_urls()
else:
    print("Crawling Wikipedia...")
    urls = crawl_wikipedia(test_url, depth)
    save_urls(urls)

print(f"Total URLs discovered: {len(urls)}")

Loading URLs from cache...
Total URLs discovered: 200


## Dictionnary

In [3]:
if dictionary_exist():
    print("Loading dictionary from cache...")
    dictionary = load_dictionary()
else:
    print("Building dictionary...")
    builder = DictionaryBuilder(urls)
    dictionary = builder.build()
    save_dictionary(dictionary)

print(f"Total unique terms: {len(dictionary)}")

Loading dictionary from cache...
Total unique terms: 49002


## Document indexing

In [4]:
print("\nIndexing dictionary...")
indexer = DocumentIndexer(dictionary)

doc_url = next(iter(urls))
doc_vectors = indexer.index_document(doc_url)
print("\n Boolean model (first 10 terms):")
for term, value in list(doc_vectors["boolean"].items())[:10]:
    print(term, value)

print("\n wf_idf model (first 10 terms):")
for term, value in list(doc_vectors["wf_idf"].items())[:10]:
    print(term, value)


Indexing dictionary...

 Boolean model (first 10 terms):
awful 0
conseil 1
noodling 0
citizen 0
comité 1
publiques 0
inspires 0
reçut 0
jardin 0
die 0

 wf_idf model (first 10 terms):
awful 0
conseil 2.3814065458906293
noodling 0
citizen 0
comité 1.7147984280919266
publiques 0
inspires 0
reçut 0
jardin 0
die 0


## Querying

In [5]:
doc_vectors = {}
for url in urls:
    doc_vectors[url] = indexer.index_document(url)

queries = [
    "religion satire",
    "intelligent design",
    "scientific criticism of religion"
]

query_indexer = QueryIndexer(dictionary)

schemes = ["boolean", "tf", "wf", "tf_idf", "wf_idf"]

for query in queries:
    print(f"\nQuery: \"{query}\"")
    query_vec = query_indexer.index_query(query)

    for scheme in schemes:
        print(f"\nTop 10 documents using {scheme}:")
        top_docs = rank_documents(query_vec, doc_vectors, scheme)

        for rank, (url, score) in enumerate(top_docs, start=1):
            print(f"{rank:2d}. {url}  (score={score:.4f})")


Query: "religion satire"

Top 10 documents using boolean:
 1. https://fr.wikipedia.org/wiki/L%27%C3%89vangile_du_monstre_en_spaghettis_volant  (score=0.0875)
 2. https://fr.wikipedia.org/wiki/Parodie_de_religion  (score=0.0724)
 3. https://fr.wikipedia.org/wiki/Recovering_from_religion  (score=0.0674)
 4. https://fr.wikipedia.org/wiki/Irr%C3%A9ligion_par_pays  (score=0.0546)
 5. https://fr.wikipedia.org/wiki/Licorne_rose_invisible  (score=0.0535)
 6. https://fr.wikipedia.org/wiki/Church_of_the_SubGenius  (score=0.0531)
 7. https://fr.wikipedia.org/wiki/Zeta_Beta_Tau  (score=0.0527)
 8. https://fr.wikipedia.org/wiki/Ath%C3%A9isme_agnostique  (score=0.0414)
 9. https://fr.wikipedia.org/wiki/Hellfire_Club  (score=0.0410)
10. https://fr.wikipedia.org/wiki/Mercredi  (score=0.0380)

Top 10 documents using tf:
 1. https://fr.wikipedia.org/wiki/Satire  (score=0.2381)
 2. https://fr.wikipedia.org/wiki/Religion  (score=0.2231)
 3. https://fr.wikipedia.org/wiki/Antireligion  (score=0.2147)
 4. h

## Language Model

In [9]:
import random
import math
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Set, Any
import re

class LanguageModel:
    def __init__(self, dictionary: Set[str]):
        self.dictionary = dictionary
        self.unigram_probs = {}  # P(t)
        self.bigram_probs = defaultdict(dict)  # P(ti|tj)
        self.documents_text = {}  # Store document text for each URL
        self.vocab_size = len(dictionary)
        self.term_to_id = {term: idx for idx, term in enumerate(dictionary)}
        self.id_to_term = {idx: term for term, idx in self.term_to_id.items()}
        
    def compute_statistics(self, documents: Dict[str, Any], 
                          indexer: DocumentIndexer, 
                          urls: List[str]):
        """
        Compute P(t) and P(ti|tj) from documents
        documents: dictionary mapping URL to document vectors (likely sparse vectors)
        indexer: DocumentIndexer instance
        urls: list of URLs to process
        """
        
        # First, collect all terms from documents
        term_counts = Counter()
        bigram_counts = defaultdict(Counter)
        total_terms = 0
        
        # For bigram counting, we need the actual text sequences
        for url in urls:
            doc_vector = documents.get(url)
            if doc_vector is None:
                continue
                
            # Handle different possible formats of doc_vector
            terms_sequence = []
            
            # Check the type of doc_vector and extract terms accordingly
            if hasattr(doc_vector, 'toarray'):  # Sparse matrix format
                # Convert sparse matrix to dense and get non-zero indices
                dense_vector = doc_vector.toarray().flatten()
                for term_id, freq in enumerate(dense_vector):
                    if freq > 0:
                        term = self.id_to_term.get(term_id)
                        if term:
                            terms_sequence.extend([term] * int(freq))
                            term_counts[term] += freq
                            total_terms += freq
                            
            elif isinstance(doc_vector, dict):  # Dictionary format {term_id: frequency}
                for term_id, freq in doc_vector.items():
                    if freq > 0:
                        term = self.id_to_term.get(term_id)
                        if term:
                            terms_sequence.extend([term] * int(freq))
                            term_counts[term] += freq
                            total_terms += freq
                            
            elif isinstance(doc_vector, list):  # List format
                # Check if it's a list of (term, freq) tuples
                if doc_vector and isinstance(doc_vector[0], (list, tuple)) and len(doc_vector[0]) == 2:
                    for term, freq in doc_vector:
                        if freq > 0:
                            terms_sequence.extend([term] * int(freq))
                            term_counts[term] += freq
                            total_terms += freq
                else:
                    # It might be a list of term IDs with frequencies embedded
                    # This is a fallback - you might need to adjust based on actual format
                    print(f"Unknown list format for {url}, skipping...")
                    continue
            else:
                print(f"Unknown document vector type for {url}: {type(doc_vector)}")
                continue
            
            # Compute bigrams from the reconstructed sequence
            # This is an approximation; ideally we'd use the original text
            for i in range(len(terms_sequence) - 1):
                current_term = terms_sequence[i]
                next_term = terms_sequence[i + 1]
                bigram_counts[current_term][next_term] += 1
        
        # Compute unigram probabilities P(t) with Laplace smoothing
        for term in self.dictionary:
            if term in term_counts:
                self.unigram_probs[term] = (term_counts[term] + 1) / (total_terms + self.vocab_size)
            else:
                self.unigram_probs[term] = 1.0 / (total_terms + self.vocab_size)
        
        # Compute bigram probabilities P(ti|tj) with Laplace smoothing
        for term1 in self.dictionary:
            total_bigrams_for_term1 = sum(bigram_counts[term1].values())
            
            for term2 in self.dictionary:
                if total_bigrams_for_term1 > 0:
                    # With Laplace smoothing
                    count = bigram_counts[term1].get(term2, 0)
                    self.bigram_probs[term1][term2] = (count + 1) / (total_bigrams_for_term1 + self.vocab_size)
                else:
                    # If term1 never appears, use unigram probability of term2 as fallback
                    self.bigram_probs[term1][term2] = self.unigram_probs.get(term2, 1.0/self.vocab_size)
    
    def generate_sentence_unigram(self, length: int = 10) -> List[str]:
        """Generate a sentence using unigram probabilities"""
        terms = list(self.dictionary)
        probs = [self.unigram_probs.get(term, 0) for term in terms]
        
        # Filter out terms with zero probability (shouldn't happen with smoothing)
        valid_terms = [(term, prob) for term, prob in zip(terms, probs) if prob > 0]
        
        if not valid_terms:
            return ["No terms available"]
        
        terms, probs = zip(*valid_terms)
        
        # Generate sentence
        sentence = random.choices(terms, weights=probs, k=length)
        return sentence
    
    def generate_sentence_bigram(self, length: int = 10) -> List[str]:
        """Generate a sentence using bigram probabilities"""
        # Start with a random term based on unigram probabilities
        terms = list(self.dictionary)
        unigram_probs = [self.unigram_probs.get(term, 1.0/len(terms)) for term in terms]
        
        current_term = random.choices(terms, weights=unigram_probs, k=1)[0]
        sentence = [current_term]
        
        # Generate subsequent terms using bigram probabilities
        for _ in range(length - 1):
            next_term_probs = self.bigram_probs[current_term]
            
            # If no valid next terms, break
            if not next_term_probs:
                break
            
            # Choose next term based on probabilities
            next_terms = list(next_term_probs.keys())
            next_probs = [next_term_probs[term] for term in next_terms]
            
            current_term = random.choices(next_terms, weights=next_probs, k=1)[0]
            sentence.append(current_term)
        
        return sentence
    
    def estimate_sentence_probability(self, sentence: List[str], model_type: str = "bigram") -> float:
        """
        Estimate the probability that a sentence was generated by this language model
        Returns log probability to avoid underflow
        """
        log_prob = 0.0
        
        if model_type == "unigram":
            for term in sentence:
                prob = self.unigram_probs.get(term, 1e-10)
                log_prob += math.log(prob)
        
        elif model_type == "bigram":
            # Start with probability of first term (using unigram)
            if sentence:
                first_term = sentence[0]
                prob_first = self.unigram_probs.get(first_term, 1e-10)
                log_prob += math.log(prob_first)
            
            # Add conditional probabilities for subsequent terms
            for i in range(len(sentence) - 1):
                current_term = sentence[i]
                next_term = sentence[i + 1]
                
                prob = self.bigram_probs[current_term].get(next_term, 1e-10)
                log_prob += math.log(prob)
        
        return log_prob


class MultiDocumentLanguageModel:
    """Handles language models for multiple documents"""
    
    def __init__(self, dictionary: Set[str]):
        self.dictionary = dictionary
        self.document_models = {}  # URL -> LanguageModel
        
    def build_models_for_documents(self, documents: Dict[str, Any],
                                  indexer: DocumentIndexer,
                                  urls: List[str]):
        """Build separate language models for each document"""
        for url in urls:
            if url in documents:
                model = LanguageModel(self.dictionary)
                # Create a model for just this document
                model.compute_statistics({url: documents[url]}, indexer, [url])
                self.document_models[url] = model
    
    def compare_sentence_across_documents(self, sentence: List[str]) -> Dict[str, float]:
        """Compare how likely a sentence is for each document model"""
        probabilities = {}
        
        for url, model in self.document_models.items():
            log_prob = model.estimate_sentence_probability(sentence, "bigram")
            probabilities[url] = log_prob
        
        # Sort by probability (higher log probability is better)
        sorted_probs = dict(sorted(probabilities.items(), 
                                  key=lambda x: x[1], 
                                  reverse=True))
        return sorted_probs


# Debug function to inspect document vector structure
def inspect_document_vector(doc_vectors, urls):
    """Helper function to understand the structure of document vectors"""
    print("\nInspecting document vector structure:")
    print("=" * 50)
    
    for url in list(urls)[:3]:  # Check first 3 documents
        vec = doc_vectors.get(url)
        print(f"\nURL: {url}")
        print(f"Type: {type(vec)}")
        
        if hasattr(vec, 'shape'):
            print(f"Shape: {vec.shape}")
        if hasattr(vec, 'toarray'):
            # If it's a sparse matrix, show sample
            dense = vec.toarray().flatten()
            non_zero = np.where(dense > 0)[0]
            print(f"Non-zero entries: {len(non_zero)}")
            if len(non_zero) > 0:
                print(f"Sample term IDs: {non_zero[:5]}")
                print(f"Sample frequencies: {dense[non_zero[:5]]}")


def demonstrate_language_models(doc_vectors, dictionary, indexer, urls):
    """Function to demonstrate the language model functionality"""
    
    print("\n" + "="*50)
    print("LANGUAGE MODEL DEMONSTRATION")
    print("="*50)
    
    # Optional: inspect structure first
    try:
        import numpy as np
        inspect_document_vector(doc_vectors, urls)
    except ImportError:
        print("NumPy not available for inspection")
    
    # Create language model for all documents combined
    print("\n1. Building combined language model...")
    combined_model = LanguageModel(dictionary)
    combined_model.compute_statistics(doc_vectors, indexer, urls)
    
    # Generate sentences
    print("\n2. Generating sentences:")
    print("-" * 30)
    
    print("\nUnigram model generated sentence:")
    unigram_sentence = combined_model.generate_sentence_unigram(8)
    print(" ".join(unigram_sentence))
    
    print("\nBigram model generated sentence:")
    bigram_sentence = combined_model.generate_sentence_bigram(8)
    print(" ".join(bigram_sentence))
    
    # Estimate probability of these sentences
    print("\n3. Sentence probabilities (log scale):")
    print("-" * 30)
    unigram_prob = combined_model.estimate_sentence_probability(unigram_sentence, "unigram")
    bigram_prob = combined_model.estimate_sentence_probability(bigram_sentence, "bigram")
    
    print(f"Unigram sentence log probability: {unigram_prob:.4f}")
    print(f"Bigram sentence log probability: {bigram_prob:.4f}")
    
    # Multi-document comparison
    print("\n4. Multi-document language models:")
    print("-" * 30)
    
    # Build models for individual documents
    multi_model = MultiDocumentLanguageModel(dictionary)
    
    # Take a subset of documents for demonstration
    sample_urls = urls[:min(5, len(urls))]
    multi_model.build_models_for_documents(doc_vectors, indexer, sample_urls)
    
    # Compare a test sentence across documents
    test_sentence = ["religion", "science", "theory", "evolution"]
    print(f"\nComparing sentence: {' '.join(test_sentence)}")
    print("\nDocument likelihoods (log probability):")
    
    doc_probs = multi_model.compare_sentence_across_documents(test_sentence)
    for url, prob in doc_probs.items():
        # Truncate URL for display
        short_url = url[:50] + "..." if len(url) > 50 else url
        print(f"{short_url}: {prob:.4f}")
    
    # Try with different sentences
    test_sentences = [
        ["intelligent", "design", "creation", "theory"],
        ["scientific", "method", "research", "evidence"],
        ["pasta", "monster", "religion", "satire"]
    ]
    
    print("\n5. Comparing multiple sentences:")
    print("-" * 30)
    
    for sentence in test_sentences:
        print(f"\nSentence: {' '.join(sentence)}")
        doc_probs = multi_model.compare_sentence_across_documents(sentence)
        
        # Show top 3 most likely documents
        top_docs = list(doc_probs.items())[:3]
        for url, prob in top_docs:
            short_url = url[:40] + "..." if len(url) > 40 else url
            print(f"  {short_url}: {prob:.4f}")


In [10]:
demonstrate_language_models(doc_vectors, dictionary, indexer, urls)


LANGUAGE MODEL DEMONSTRATION

Inspecting document vector structure:

URL: https://fr.wikipedia.org/wiki/%C3%89vang%C3%A9lisation
Type: <class 'dict'>

URL: https://fr.wikipedia.org/wiki/Religion
Type: <class 'dict'>

URL: https://fr.wikipedia.org/wiki/Intelligent_design_movement
Type: <class 'dict'>

1. Building combined language model...


TypeError: '>' not supported between instances of 'dict' and 'int'

In [ ]:
# ---- language model ------
import math
import random
from collections import Counter, defaultdict
import requests
from bs4 import BeautifulSoup
import re

# ----------- Text Processing -------------

def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ\s]", " ", text)
    return [t for t in text.split() if len(t) > 1]

def fetch_text(url):
    html = requests.get(url, timeout=10).text
    soup = BeautifulSoup(html, "html.parser")

    content = soup.find("div", {"id": "mw-content-text"})
    if content:
        return content.get_text()
    return ""

# ----------- Language Model Estimation -------------

def compute_unigram_model(tokens):
    total = len(tokens)
    counts = Counter(tokens)
    return {t: c / total for t, c in counts.items()}

def compute_bigram_model(tokens):
    bigram_counts = defaultdict(Counter)
    for w1, w2 in zip(tokens[:-1], tokens[1:]):
        bigram_counts[w1][w2] += 1

    bigram_probs = {}
    for w1 in bigram_counts:
        total = sum(bigram_counts[w1].values())
        bigram_probs[w1] = {
            w2: c / total for w2, c in bigram_counts[w1].items()
        }

    return bigram_probs

# ----------- Sampling -------------

def sample_from_distribution(distribution):
    r = random.random()
    acc = 0
    for word, prob in distribution.items():
        acc += prob
        if acc >= r:
            return word
    return random.choice(list(distribution.keys()))

def generate_sentence_unigram(unigram_model, length=15):
    return " ".join(sample_from_distribution(unigram_model) for _ in range(length))

def generate_sentence_bigram(bigram_model, length=15):
    word = random.choice(list(bigram_model.keys()))
    sentence = [word]

    for _ in range(length - 1):
        if word not in bigram_model:
            break
        word = sample_from_distribution(bigram_model[word])
        sentence.append(word)

    return " ".join(sentence)

# ----------- Sentence Probability -------------

def sentence_probability(sentence, unigram_model, bigram_model=None):
    tokens = clean_and_tokenize(sentence)

    log_prob = 0.0
    eps = 1e-10  # smoothing

    if bigram_model:
        for w1, w2 in zip(tokens[:-1], tokens[1:]):
            prob = bigram_model.get(w1, {}).get(w2, eps)
            log_prob += math.log(prob)
    else:
        for w in tokens:
            prob = unigram_model.get(w, eps)
            log_prob += math.log(prob)

    return log_prob

# ----------- Build Models for Documents -------------

doc_models = {}

print("\nBuilding language models...")

for url in list(urls):
    text = fetch_text(url)
    tokens = clean_and_tokenize(text)

    if len(tokens) < 50:
        continue   # skip tiny / empty documents

    unigram = compute_unigram_model(tokens)
    bigram = compute_bigram_model(tokens)

    doc_models[url] = (unigram, bigram)

print(f"Built models for {len(doc_models)} documents.")

# ----------- Generate Sentences -------------

print("\nSentence generation (Unigram):")
u_model, b_model = list(doc_models.values())[0]

for i in range(3):
    print("-", generate_sentence_unigram(u_model))

print("\nSentence generation (Bigram):")
for i in range(3):
    print("-", generate_sentence_bigram(b_model))

# ----------- Sentence Classification -------------

test_sentence = "religion scientifique critique satire"

print(f"\nEvaluating sentence: \"{test_sentence}\"")

scores = []
for url, (u_model, b_model) in doc_models.items():
    score = sentence_probability(test_sentence, u_model, b_model)
    scores.append((url, score))

scores.sort(key=lambda x: x[1], reverse=True)

print("\nMost likely document models:")
for rank, (url, score) in enumerate(scores[:], start=1):
    print(f"{rank:2d}. {url}   logP={score:.4f}")


Building language models...


C:\Users\ferna\AppData\Local\Temp\ipykernel_13336\913121843.py:18: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(html, "html.parser")


Built models for 0 documents.

Sentence generation (Unigram):


IndexError: list index out of range